In [101]:
email_conversation = """From: jiwon jeong (jwjw9603@skku.edu)
To: 돌멩e 대리님 (dolmeng@mail.net)
Subject: 방송 방제 관련 문의

안녕하세요, 돌멩e 대리님,

저는 jiwon jeong입니다. 최근 방송에서 방제 관련 내용을 다루고 있어서 몇 가지 궁금한 점이 생겨 이렇게 이메일을 드립니다.
1. 방송에서 다룬 방제 방법 중 가장 효과적인 방법은 무엇인가요?
2. 방제 시 주의해야 할 점이나 팁이 있다면 알려주실 수 있을까요?
3. 추가로 참고할 만한 자료나 링크가 있다면 공유 부탁드립니다.   

jiwon jeong 드림"""

In [102]:
# 이메일 본문으로부터 주요 엔티티 추출
from pydantic import BaseModel, Field


class EmailSummary(BaseModel):
    sender_name: str = Field(description="이메일을 보낸 사람")
    sender_email: str = Field(description="이메일을 보낸 사람의 이메일 주소")
    recipient_name: str = Field(description="이메일을 받은 사람")
    recipient_email: str = Field(description="이메일을 받은 사람의 이메일 주소")
    subject: str = Field(description="이메일 제목")
    meeting_date: str = Field(description="제안된 미팅 날짜")
    meeting_time: str = Field(description="제안된 미팅 시간")
    meeting_location: str = Field(description="제안된 미팅 장소")
    summary: str = Field(description="이메일의 주요 내용 요약")

In [103]:
# LCEL 구조

# chain = prompt | llm | output_parser

In [104]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)
output_parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [105]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
    You are a helpful assistant. Please answer the following questions in KOREAN.
    
    #QUESTION:
    다음의 이메일 내용 중에서 주요 내용을 추출해 주세요.
    
    #EMAIL CONVERSATION:
    {email_conversation}
    
    #FORMAT:
    {format}
    """
)
prompt = prompt.partial(format=output_parser.get_format_instructions())

In [106]:
# 체인 생성
chain = prompt | llm | output_parser

# 체인 실행
answer = chain.invoke({"email_conversation": email_conversation})
print(answer)

sender_name='jiwon jeong' sender_email='jwjw9603@skku.edu' recipient_name='돌멩e 대리님' recipient_email='dolmeng@mail.net' subject='방송 방제 관련 문의' meeting_date='' meeting_time='' meeting_location='' summary='jiwon jeong은 방송에서 다룬 방제 방법 중 가장 효과적인 방법, 방제 시 주의해야 할 점이나 팁, 추가로 참고할 만한 자료나 링크에 대해 문의하고 있습니다.'


In [107]:
answer.summary

'jiwon jeong은 방송에서 다룬 방제 방법 중 가장 효과적인 방법, 방제 시 주의해야 할 점이나 팁, 추가로 참고할 만한 자료나 링크에 대해 문의하고 있습니다.'

In [108]:
answer

EmailSummary(sender_name='jiwon jeong', sender_email='jwjw9603@skku.edu', recipient_name='돌멩e 대리님', recipient_email='dolmeng@mail.net', subject='방송 방제 관련 문의', meeting_date='', meeting_time='', meeting_location='', summary='jiwon jeong은 방송에서 다룬 방제 방법 중 가장 효과적인 방법, 방제 시 주의해야 할 점이나 팁, 추가로 참고할 만한 자료나 링크에 대해 문의하고 있습니다.')

## 검색: SERP API

In [ ]:
import os

os.environ["SERPAPI_API_KEY"] = (
    "YOUR_API_KEY_HERE"
)

In [149]:
from langchain_community.utilities import SerpAPIWrapper

params = {"engine": "google", "gl": "kr", "hl": "ko", "num": "3"}

search = SerpAPIWrapper(params=params)

In [150]:
print(search)

search_engine=<class 'serpapi.google_search.GoogleSearch'> params={'engine': 'google', 'gl': 'kr', 'hl': 'ko', 'num': '3'} serpapi_api_key='03ca25425895f89043e7ac1fe92b334f47d71ffc6a7392070bdc1f09dbec6aee' aiosession=None


In [151]:
# 내가 모르는 사람으로부터 이메일을 받았을 때, 그 사람에 대한 추가 정보를 검색

query = f"{answer.sender_email} {answer.sender_name}"
query

'jwjw9603@skku.edu jiwon jeong'

In [152]:
search_result = search.run(query)

In [153]:
print(search_result)

['Contact : jwjw9603@g.skku.edu I am interested in Natural Language Processing and Commonsense Reasoning for the Next of QnA System. 할 수 있다!!!', 'Jiwon Jeong. Sungkyunkwan University (SKKU). g.skku.edu의 이메일 확인됨. Natural Language ProcessingLLMsLogical Fallacy. 학술자료인용. 제목. 정렬.', '• Jimin An, als398@skku.edu, Master, 2024. • Jiwon Jeong, jwjw9603@g.skku.edu, Master, 2024. • JunKoo Lee, dlwnsrn0727@g.skku.edu, Master, 2024. • HyunSung Kim ...']


In [154]:
print(type(search_result))

<class 'str'>


In [155]:
eval(search_result)  # 리스트 형태로 변환

['Contact : jwjw9603@g.skku.edu I am interested in Natural Language Processing and Commonsense Reasoning for the Next of QnA System. 할 수 있다!!!',
 'Jiwon Jeong. Sungkyunkwan University (SKKU). g.skku.edu의 이메일 확인됨. Natural Language ProcessingLLMsLogical Fallacy. 학술자료인용. 제목. 정렬.',
 '• Jimin An, als398@skku.edu, Master, 2024. • Jiwon Jeong, jwjw9603@g.skku.edu, Master, 2024. • JunKoo Lee, dlwnsrn0727@g.skku.edu, Master, 2024. • HyunSung Kim ...']

In [156]:
search_result = eval(search_result)
search_result

['Contact : jwjw9603@g.skku.edu I am interested in Natural Language Processing and Commonsense Reasoning for the Next of QnA System. 할 수 있다!!!',
 'Jiwon Jeong. Sungkyunkwan University (SKKU). g.skku.edu의 이메일 확인됨. Natural Language ProcessingLLMsLogical Fallacy. 학술자료인용. 제목. 정렬.',
 '• Jimin An, als398@skku.edu, Master, 2024. • Jiwon Jeong, jwjw9603@g.skku.edu, Master, 2024. • JunKoo Lee, dlwnsrn0727@g.skku.edu, Master, 2024. • HyunSung Kim ...']

In [157]:
search_result_string = "\n".join(search_result)
search_result_string

'Contact : jwjw9603@g.skku.edu I am interested in Natural Language Processing and Commonsense Reasoning for the Next of QnA System. 할 수 있다!!!\nJiwon Jeong. Sungkyunkwan University (SKKU). g.skku.edu의 이메일 확인됨. Natural Language ProcessingLLMsLogical Fallacy. 학술자료인용. 제목. 정렬.\n• Jimin An, als398@skku.edu, Master, 2024. • Jiwon Jeong, jwjw9603@g.skku.edu, Master, 2024. • JunKoo Lee, dlwnsrn0727@g.skku.edu, Master, 2024. • HyunSung Kim ...'

In [158]:
from langchain_core.prompts import PromptTemplate

report_prompt = PromptTemplate.from_template(
    """당신은 이메일의 주요 정보를 바탕으로 요약 정리해 주는 전문가 입니다.
당신의 임무는 다음의 이메일 정보를 바탕으로 보고서 형식의 요약을 작성하는 것입니다.
주어진 정보를 기반으로 양식(format)에 맞추어 요약을 작성해 주세요.

#Information:
- Sender: {sender}
- Additional Information about sender: {additional_information}
- Recipient: {recipient}
- Subject: {subject}
- Meeting Date: {meeting_date}
- Meeting Time: {meeting_time}
- Meeting Location: {meeting_location}
- Summary: {summary}

#FORMAT(in markdown format):
🙇‍♂️ 보낸 사람:
- (보낸 사람의 이름, 회사 정보)

📧 이메일 주소:
- (보낸 사람의 이메일 주소)

😍 보낸 사람과 관련하여 검색된 추가 정보:
- (검색된 추가 정보)

✅ 주요 내용:
- (이메일 제목, 요약)

⏰ 일정:
- (미팅 날짜 및 시간)

#Answer:"""
)

In [159]:
from langchain_core.output_parsers import StrOutputParser

report_chain = (
    report_prompt | ChatOpenAI(model="gpt-4-turbo", temperature=0) | StrOutputParser()
)

report_response = report_chain.invoke(
    {
        "sender": answer.sender_name,
        "additional_information": search_result_string,
        "recipient": answer.recipient_name,
        "subject": answer.subject,
        "meeting_date": answer.meeting_date,
        "meeting_time": answer.meeting_time,
        "meeting_location": answer.meeting_location,
        "summary": answer.summary,
    }
)

In [160]:
print(report_response)

🙇‍♂️ 보낸 사람:
- Jiwon Jeong, Sungkyunkwan University (SKKU)

📧 이메일 주소:
- jwjw9603@g.skku.edu

😍 보낸 사람과 관련하여 검색된 추가 정보:
- Jiwon Jeong은 자연어 처리 및 상식 추론에 관심이 있으며, QnA 시스템의 다음 단계에 대한 연구를 진행 중입니다. 그는 성균관대학교에서 석사 과정을 밟고 있으며, 자연어 처리, LLMs, 논리적 오류에 대한 학술 자료를 인용하고 있습니다.

✅ 주요 내용:
- 제목: 방송 방제 관련 문의
- 요약: Jiwon Jeong은 방송에서 다룬 방제 방법 중 가장 효과적인 방법, 방제 시 주의해야 할 점이나 팁, 추가로 참고할 만한 자료나 링크에 대해 문의하고 있습니다.

⏰ 일정:
- 미팅 날짜 및 시간: 정보 없음
